# Chapitre 3 - Cours 1 : Journaux append-only et index de hachage

Ce notebook ouvre le chapitre *Storage and Retrieval* par le point de vue de la base de
données : comment ranger les données qu'on lui confie, et comment les retrouver ? Il
couvre le début de la section *Data Structures That Power Your Database* et la
sous-section *Hash Indexes* : la base « bash » en deux fonctions, la notion d'index et
son coût, l'index de hachage en mémoire à la Bitcask, les segments, leur fusion et leur
compaction, et les détails qui font un vrai moteur (suppressions, reprise après panne,
écritures partielles, concurrence).

**Plan.**

1. La base de données la plus simple du monde : un fichier où l'on ajoute des lignes.
2. Qu'est-ce qu'un index ? Le compromis fondamental lecture contre écriture.
3. Un index de hachage en mémoire au-dessus du journal.
4. Segments, fusion et compaction : récupérer l'espace des valeurs écrasées.
5. Ce qui fait un vrai moteur : format binaire, suppressions, pannes, concurrence.
6. Pourquoi ajouter plutôt que modifier en place ? Et les limites de l'index de hachage.

In [1]:
import os
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import storage as S
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
WORK = tempfile.mkdtemp(prefix="ddia_stockage_")